## Author: Kenny Loh Kit Yi

In [1]:
import os
import shutil
from classes.data_crawler import DataCrawler
from classes.web_scraper import WebScraper
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from classes.dataframe_saver import DataFrameSaver
import subprocess

In [2]:
# Delete old HDFS version
subprocess.run(['hdfs', 'dfs', '-rm', 'articles/*'])

Deleted articles/_SUCCESS
Deleted articles/articles.csv


CompletedProcess(args=['hdfs', 'dfs', '-rm', 'articles/*'], returncode=0)

In [3]:
# Initialize Spark session
spark = SparkSession \
    .builder \
    .appName("Assignment") \
    .getOrCreate()

24/12/22 21:15:50 WARN Utils: Your hostname, LAPTOP-PFPL3CLD. resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
24/12/22 21:15:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/12/22 21:15:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/12/22 21:15:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/12/22 21:15:52 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:
# Web scraping logic
web_url = "https://www.sinarharian.com.my"
category_links = DataCrawler.extract_category_links(web_url)
max_articles = 1
articles = WebScraper.extract_all_articles(web_url, category_links, max_articles)

Processing category: https://www.sinarharian.com.my/nasional


In [5]:
# Create DataFrame
df = spark.createDataFrame([(article,) for article in articles], ["article"])
output_dir = "/home/student/de-assgt/content"
file_name = "articles.csv"

DataFrameSaver.save_to_csv(df, output_dir, file_name)

print(f"Article content saved as {file_name} in {output_dir}")

Article content saved as articles.csv in /home/student/de-assgt/content


In [6]:
hdfs_path = "hdfs://localhost:9000/user/student/articles"
df.coalesce(1).write.csv(hdfs_path, header=True, mode="overwrite")
print(f"Article content saved to HDFS at: {hdfs_path}")

Article content saved to HDFS at: hdfs://localhost:9000/user/student/articles


In [7]:
# Rename part-00000 file
subprocess.run(['hdfs', 'dfs', '-mv', 'articles/part-00000*', 'articles/articles.csv'])

CompletedProcess(args=['hdfs', 'dfs', '-mv', 'articles/part-00000*', 'articles/articles.csv'], returncode=0)

In [8]:
spark.stop()